# In-Depth Guide: Prediction

This notebook covers how to use a trained **detectree2** model to make predictions on large-scale orthomosaics, including tiling for prediction, running inference, and the full postprocessing pipeline (`stitch`, `clean`, `post_clean`).

For the full tutorial, see the [documentation](https://patball1.github.io/detectree2/tutorials/04_prediction.html).

## Setup

In [2]:
!pip install torch torchvision torchaudio
!pip install 'git+https://github.com/facebookresearch/detectron2.git'
!pip install detectree2

  Cloning https://github.com/facebookresearch/detectron2.git to /tmp/pip-req-build-v41blbqp
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2.git /tmp/pip-req-build-v41blbqp
  Resolved https://github.com/facebookresearch/detectron2.git to commit 02b5c4e295e990042a714712c21dc79b731e8833
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 29.6 MB/s eta 0:00:00
  Created wheel for detectron2: filename=detectron2-0.6-cp312-cp312-linux_x86_64.whl size=7105783 sha2

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Tiling the Orthomosaic for Prediction

Tile the entire orthomosaic so that a crown map can be made for the full landscape. Tiles should be approximately the same size as those trained on (~100 m). A buffer (here 30 m) is included so we can discard partial crowns at tile edges.

In [8]:
from detectree2.preprocessing.tiling import tile_data
from detectree2.models.outputs import project_to_geojson, stitch_crowns, clean_crowns, post_clean
from detectree2.models.predict import predict_on_data
from detectree2.models.train import setup_cfg
from detectron2.engine import DefaultPredictor
import rasterio

In [11]:
site_path = "/content/sample_data"
img_path = site_path + "/orto.tif"
tiles_path = site_path + "/tilespred/"

# Define tiling parameters
buffer = 30 # Overlap between tiles
tile_width = 40 # Width of each tile
tile_height = 40 # Height of each tile

# Call the tile_data function
tile_data(img_path, tiles_path, buffer, tile_width, tile_height, dtype_bool=True)

  0%|          | 0/48 [00:00<?, ?it/s]

ERROR:detectree2.preprocessing.tiling:RasterioIOError while applying mask [{'type': 'Polygon', 'coordinates': [[[670879.0, 3559097.0], [670879.0, 3559197.0], [670779.0, 3559197.0], [670779.0, 3559097.0], [670879.0, 3559097.0]]]}]: Read failed. See previous exception for details.
ERROR:detectree2.preprocessing.tiling:RasterioIOError while applying mask [{'type': 'Polygon', 'coordinates': [[[670879.0, 3559137.0], [670879.0, 3559237.0], [670779.0, 3559237.0], [670779.0, 3559137.0], [670879.0, 3559137.0]]]}]: Read failed. See previous exception for details.
ERROR:detectree2.preprocessing.tiling:RasterioIOError while applying mask [{'type': 'Polygon', 'coordinates': [[[670879.0, 3559177.0], [670879.0, 3559277.0], [670779.0, 3559277.0], [670779.0, 3559177.0], [670879.0, 3559177.0]]]}]: Read failed. See previous exception for details.
ERROR:detectree2.preprocessing.tiling:RasterioIOError while applying mask [{'type': 'Polygon', 'coordinates': [[[670879.0, 3559217.0], [670879.0, 3559317.0], [6

In [12]:
site_path = "/content/sample_data"
img_path = site_path + "/orto.tif"
tiles_path = site_path + "/tilespred/"

# Specify tiling
buffer = 30
tile_width = 40
tile_height = 40
tile_data(img_path, tiles_path, buffer, tile_width, tile_height, dtype_bool=True)

  0%|          | 0/48 [00:00<?, ?it/s]

ERROR:detectree2.preprocessing.tiling:RasterioIOError while applying mask [{'type': 'Polygon', 'coordinates': [[[670879.0, 3559097.0], [670879.0, 3559197.0], [670779.0, 3559197.0], [670779.0, 3559097.0], [670879.0, 3559097.0]]]}]: Read failed. See previous exception for details.
ERROR:detectree2.preprocessing.tiling:RasterioIOError while applying mask [{'type': 'Polygon', 'coordinates': [[[670879.0, 3559137.0], [670879.0, 3559237.0], [670779.0, 3559237.0], [670779.0, 3559137.0], [670879.0, 3559137.0]]]}]: Read failed. See previous exception for details.
ERROR:detectree2.preprocessing.tiling:RasterioIOError while applying mask [{'type': 'Polygon', 'coordinates': [[[670879.0, 3559177.0], [670879.0, 3559277.0], [670779.0, 3559277.0], [670779.0, 3559177.0], [670879.0, 3559177.0]]]}]: Read failed. See previous exception for details.
ERROR:detectree2.preprocessing.tiling:RasterioIOError while applying mask [{'type': 'Polygon', 'coordinates': [[[670879.0, 3559217.0], [670879.0, 3559317.0], [6

The `RasterioIOError: Read failed` suggests an issue with the input GeoTIFF file. Let's verify its existence and try to open it directly with `rasterio` to get more diagnostic information.

In [14]:
import os
import rasterio

img_path = "/content/sample_data/orto.tif"

# Check if the file exists
if not os.path.exists(img_path):
    print(f"Error: The file '{img_path}' does not exist.")
else:
    print(f"File '{img_path}' exists. Attempting to open with rasterio...")
    try:
        with rasterio.open(img_path) as src:
            print(f"Successfully opened '{img_path}'.")
            print(f"Image dimensions: {src.width}x{src.height}")
            print(f"Number of bands: {src.count}")
            print(f"CRS: {src.crs}")
            print("This indicates the file is likely valid and readable.")
    except rasterio.errors.RasterioIOError as e:
        print(f"Error opening '{img_path}' with rasterio: {e}")
        print("This might indicate a corrupted file or an unsupported GeoTIFF format.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

File '/content/sample_data/orto.tif' exists. Attempting to open with rasterio...
Successfully opened '/content/sample_data/orto.tif'.
Image dimensions: 20014x17053
Number of bands: 4
CRS: EPSG:32636
This indicates the file is likely valid and readable.


> **Warning:** If tiles are outputting as blank images, set `dtype_bool=True`. Avoid supplying crown polygons otherwise the function will run as if tiling for training.

### Downloading a Pre-trained Model

You can download a pre-trained model from the model garden:

In [13]:
!wget https://zenodo.org/records/15863800/files/250312_flexi.pth

--2026-07-04 19:41:12--  https://zenodo.org/records/15863800/files/250312_flexi.pth
Resolving zenodo.org (zenodo.org)... 188.185.48.75, 137.138.52.235, 137.138.153.219, ...
Connecting to zenodo.org (zenodo.org)|188.185.48.75|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 503063232 (480M) [application/octet-stream]
Saving to: ‘250312_flexi.pth’

250312_flexi.pth    100%[===================>] 479.76M  12.7MB/s    in 39s     

2026-07-04 19:41:52 (12.3 MB/s) - ‘250312_flexi.pth’ saved [503063232/503063232]



## 2. Running Predictions

In [ ]:
trained_model = "./230103_randresize_full.pth"
cfg = setup_cfg(update_model=trained_model)
predict_on_data(tiles_path, predictor=DefaultPredictor(cfg))

## 3. Postprocessing

Project predictions back into geographic space, then stitch and clean.

In [ ]:
# Project tile predictions to geo-referenced crowns
project_to_geojson(tiles_path, tiles_path + "predictions/", tiles_path + "predictions_geo/")

In [ ]:
# Stitch crowns together, handling overlaps in the buffer
crowns = stitch_crowns(tiles_path + "predictions_geo/", 1)

# Clean overlapping predictions (removes lower-confidence duplicates)
clean = clean_crowns(crowns, 0.6, confidence=0)  # set confidence>0 to filter less confident crowns

Optionally filter by confidence score. `Confidence_score` is a column in the crowns GeoDataFrame and is a tunable parameter.

In [ ]:
clean = clean[clean["Confidence_score"] > 0.5]

## 4. Filling Gaps with `post_clean`

`clean_crowns` aggressively removes overlapping predictions, which can leave gaps. `post_clean` iteratively fills these gaps by adding back crowns from the original (unclean) set that do not significantly overlap with the cleaned crowns. It uses a lower IoU threshold (default 0.3) and repeats until no new crowns are added.

In [ ]:
clean = post_clean(crowns, clean, iou_threshold=0.3)

You can control the maximum number of iterations (default 5):

In [ ]:
# Alternative with explicit max iterations
# clean = post_clean(crowns, clean, iou_threshold=0.3, max_iterations=3)

## 5. Simplify and Save

Crown polygons have many vertices (generated from pixelwise masks). Simplify for easier editing in GIS software. The `tolerance` has the same units as the CRS (meters for UTM).

In [ ]:
clean = clean.set_geometry(clean.simplify(0.3))

# Save the crown map
clean.to_file(site_path + "/crowns_out.gpkg")

You can now view `crowns_out.gpkg` in QGIS or ArcGIS.